# 06 数据集快速概览

**用途**：一键展示数据集全貌，包括字段 schema、样本数、tag 分布、GT 标注统计、图片尺寸、预测字段统计。


In [8]:
# ── 0. 配置区 ──────────────────────────────────────────────
from pathlib import Path

DATASET_NAME = "sahi_null_v2_ms1_0605-0621_40_ok"  # 目标数据集名
GT_FIELD     = "ground_truth"                       # GT 字段名
PRED_FIELD   = "small_slices_a03_yolo11s_custom7null_cv1_ms2_0809-0823_10_ok_16"                                   # 预测字段名，留空则跳过预测统计


In [2]:
# ── 1. 初始化日志 ──────────────────────────────────────────
import logging
import ipynbname

log_file_name = ipynbname.name() + ".log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file_name, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)
logger.info("============ 06_dataset_overview 初始化完成 ============")


2026-02-19 16:13:42,402 [INFO] ============ 06_dataset_overview 初始化完成 ============


## Step 1: 加载数据集，展示字段 schema


In [3]:
import fiftyone as fo
import pandas as pd
from IPython.display import display

logger.info("Step 1 开始：加载数据集")

ds = fo.load_dataset(DATASET_NAME)
print(f"数据集名称: {ds.name}")
print(f"样本总数: {len(ds)}")

schema = ds.get_field_schema()
schema_rows = [{"field_name": k, "field_type": str(v)} for k, v in schema.items()]
schema_df = pd.DataFrame(schema_rows)
display(schema_df)

logger.info(f"Step 1 完成：共 {len(ds)} 个样本，{len(schema)} 个字段")


2026-02-19 16:13:46,070 [INFO] Step 1 开始：加载数据集


数据集名称: sahi_null_v2_ms1_0605-0621_40_ok
样本总数: 243


,field_name,field_type
0,id,fiftyone.core.fields.ObjectIdField
1,filepath,fiftyone.core.fields.StringField
2,tags,fiftyone.core.fields.ListField(fiftyone.core.f...
3,metadata,fiftyone.core.fields.EmbeddedDocumentField(fif...
4,created_at,fiftyone.core.fields.DateTimeField
5,last_modified_at,fiftyone.core.fields.DateTimeField
6,ground_truth,fiftyone.core.fields.EmbeddedDocumentField(fif...
7,small_slices_a03_yolo11s_custom7null_cv1_ms2_0...,fiftyone.core.fields.EmbeddedDocumentField(fif...
8,small_slices_a03_yolo11n_custom7null_cv1_ms2_0...,fiftyone.core.fields.EmbeddedDocumentField(fif...
9,small_slices_a03_yolo11s_custom7null_cv1_ms2_0...,fiftyone.core.fields.EmbeddedDocumentField(fif...


2026-02-19 16:13:47,545 [INFO] Step 1 完成：共 243 个样本，15 个字段


## Step 2: Tag 分布


In [4]:
logger.info("Step 2 开始：统计 Tag 分布")

tag_counts = ds.count_values("tags")
if tag_counts:
    tag_df = pd.DataFrame(list(tag_counts.items()), columns=["tag", "count"]).sort_values("count", ascending=False)
    display(tag_df)
else:
    print("（无 tags）")

logger.info(f"Step 2 完成：共 {len(tag_counts)} 种 tag")


2026-02-19 16:14:18,178 [INFO] Step 2 开始：统计 Tag 分布
2026-02-19 16:14:18,231 [INFO] Step 2 完成：共 0 种 tag


（无 tags）


## Step 3: GT 标注统计


In [5]:
logger.info("Step 3 开始：GT 标注统计")

from fiftyone import ViewField as F

if GT_FIELD and GT_FIELD in ds.get_field_schema():
    gt_counts = ds.values(f"{GT_FIELD}.detections")
    gt_count_list = [len(d) if d else 0 for d in gt_counts]
    gt_series = pd.Series(gt_count_list, name="gt_det_count")

    print(f"有 GT 标注的样本数: {(gt_series > 0).sum()} / {len(ds)}")
    print(f"平均每图标注数: {gt_series.mean():.2f}")
    print(f"最多标注数: {gt_series.max()}")
    display(gt_series.value_counts().sort_index().head(20).rename("样本数").to_frame())
else:
    print(f"GT_FIELD='{GT_FIELD}' 不存在于数据集中，跳过")

logger.info("Step 3 完成")


2026-02-19 16:14:27,877 [INFO] Step 3 开始：GT 标注统计


有 GT 标注的样本数: 243 / 243
平均每图标注数: 3.50
最多标注数: 12


,样本数
gt_det_count,
1,74
2,31
3,41
4,32
5,14
6,15
7,13
8,9
9,6


2026-02-19 16:14:27,997 [INFO] Step 3 完成


## Step 4: 图片尺寸


In [6]:
logger.info("Step 4 开始：图片尺寸统计")

ds.compute_metadata()
widths  = ds.values("metadata.width")
heights = ds.values("metadata.height")

widths  = [w for w in widths if w is not None]
heights = [h for h in heights if h is not None]

size_df = pd.DataFrame({"width": widths, "height": heights})
display(size_df.describe().round(1))

logger.info(f"Step 4 完成：宽度范围 {min(widths)}~{max(widths)}")


2026-02-19 16:18:03,262 [INFO] Step 4 开始：图片尺寸统计


,width,height
count,243.0,243.0
mean,4656.0,3496.0
std,0.0,0.0
min,4656.0,3496.0
25%,4656.0,3496.0
50%,4656.0,3496.0
75%,4656.0,3496.0
max,4656.0,3496.0


2026-02-19 16:18:03,271 [INFO] Step 4 完成：宽度范围 4656~4656


## Step 5: 预测字段统计（PRED_FIELD 非空时）


In [9]:
logger.info("Step 5 开始：预测字段统计")

if PRED_FIELD and PRED_FIELD in ds.get_field_schema():
    pred_confs = []
    for dets in ds.values(f"{PRED_FIELD}.detections"):
        if dets:
            for d in dets:
                if d.confidence is not None:
                    pred_confs.append(d.confidence)

    conf_series = pd.Series(pred_confs, name="confidence")
    bins = [0, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0]
    binned = pd.cut(conf_series, bins=bins)
    display(binned.value_counts().sort_index().rename("检测数").to_frame())
    logger.info(f"Step 5 完成：共 {len(pred_confs)} 个预测框")
else:
    print(f"PRED_FIELD='{PRED_FIELD}' 为空或不存在，跳过预测统计")
    logger.info("Step 5 跳过：PRED_FIELD 未配置")


2026-02-19 16:18:58,310 [INFO] Step 5 开始：预测字段统计


,检测数
confidence,
"(0.0, 0.3]",401
"(0.3, 0.5]",228
"(0.5, 0.7]",252
"(0.7, 0.8]",177
"(0.8, 0.9]",428
"(0.9, 0.95]",968
"(0.95, 1.0]",0


2026-02-19 16:18:58,400 [INFO] Step 5 完成：共 2454 个预测框


## 打开 App


In [ ]:
session = fo.launch_app(ds)
